# E1 final expanded-data H0 reference

Train the hierarchy-only H0 baseline on the frozen tensor cohort: official
train, validation, and test members from the required number of archives,
plus all unique exemplars. The run uses
the same manifest generated from the current tensor index; no historical
split manifest is loaded or used.

The notebook is resumable within and across Kaggle sessions. Each run has an
independent time limit, optimizer backup, best checkpoint, result directory,
and provenance signature. The shared result archive is refreshed after every
run. A timeout evaluates the best checkpoint seen so far; a later session then
continues from the periodic optimizer backup.

This is the E1 in-distribution reference from the September 3 thesis plan. It
is not the architecture-grouped structural comparison: any official-split
group conflicts are audited and recorded rather than silently repaired.


In [ ]:
from pathlib import Path
import hashlib
import json
import os
import re
import shutil
import subprocess
import sys
import time
import torch

WORK = Path("/kaggle/working")
REPO_DIR = WORK / "hls-surrogate-lab"
RESULTS_DIR = WORK / "results"
CONFIG_DIR = WORK / "configs"

REPO_URL = "https://github.com/brios-polimi/hls-surrogate-lab.git"
REPO_REF = "4db75266efcdc2f7c122f7919713e2bad267dcc2"
TENSOR_REPO_ID = "BrendanRios/wa-hls4ml-tensors-hierarchical"
TENSOR_REVISION = "9fa339c2a26d36f0bda19ae7e79bc2748ad8e0f3"
# Keep the Hugging Face snapshot outside /kaggle/working; the tensor
# cohort is too large to fit there alongside the repository and results.
HF_CACHE_DIR = Path("/tmp/wa_hls4ml_hierarchy_hf_cache")
DOWNLOAD_HEARTBEAT_SECONDS = 60
DOWNLOAD_STALL_WARNING_MINUTES = 5

# huggingface_hub reads these when it is imported in the download cell.
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "0"
os.environ["HF_HOME"] = str(HF_CACHE_DIR)
os.environ["HF_HUB_CACHE"] = str(HF_CACHE_DIR / "hub")
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"
os.environ["HF_XET_CACHE"] = str(HF_CACHE_DIR / "xet")

# Attach the zip produced by the earlier session. Leave this as None to
# auto-find exactly one matching archive under /kaggle/input.
PREVIOUS_RESULTS_ROOT = None

# Keep True after the sparse-relation DDP fix is in REPO_REF. Set False only
# as a one-GPU fallback when testing an older repository revision.
USE_DDP = True
EFFECTIVE_GLOBAL_BATCH = 16
# E1 baseline contract from the September 3 thesis execution plan.
ACTIVE_RUNS = ["hierarchical"]
TRAIN_BUDGETS = {
    "hierarchical": "11h",
}
SEED = 42
EXPECTED_MAIN_ARCHIVES = 32
BASELINE_ARCHIVES = 4
SCALE_PERCENT = 100 * EXPECTED_MAIN_ARCHIVES // BASELINE_ARCHIVES
STUDY_ID = f"e1_h0_final_reference_a{EXPECTED_MAIN_ARCHIVES}"
ARCHIVE_NAME = f"hls_surrogate_lab_{STUDY_ID}_results"
KERNEL_TYPES = [
    "2layer", "3layer", "conv1d", "conv2d",
    "dense_latency", "dense_resource", "rule4ml",
]


def assert_commit(value, name):
    assert re.fullmatch(r"[0-9a-fA-F]{40,64}", value), (
        f"{name} must be an immutable full commit hash; got {value!r}"
    )


assert_commit(REPO_REF, "REPO_REF")
assert_commit(TENSOR_REVISION, "TENSOR_REVISION")
assert EXPECTED_MAIN_ARCHIVES > 0
assert BASELINE_ARCHIVES > 0
assert (100 * EXPECTED_MAIN_ARCHIVES) % BASELINE_ARCHIVES == 0
assert ACTIVE_RUNS == ["hierarchical"], ACTIVE_RUNS
for run_name, budget in TRAIN_BUDGETS.items():
    assert re.fullmatch(r"[1-9][0-9]*[smhd]", budget), (run_name, budget)

GPU_COUNT = torch.cuda.device_count()
assert GPU_COUNT >= 1, "Enable a Kaggle GPU accelerator before running."
if USE_DDP:
    assert EFFECTIVE_GLOBAL_BATCH % GPU_COUNT == 0, (
        "Effective global batch must divide evenly across GPUs"
    )
PRECISION = "bf16" if torch.cuda.is_bf16_supported() else "float32"
print("PyTorch:", torch.__version__)
print("CUDA devices:", GPU_COUNT)
for index in range(GPU_COUNT):
    print(index, torch.cuda.get_device_name(index))
print("Precision:", PRECISION)
print("DDP world size:", GPU_COUNT if USE_DDP else 1)
print("Effective global batch:", EFFECTIVE_GLOBAL_BATCH)
print("Active runs:", ACTIVE_RUNS)
print("Training budgets:", TRAIN_BUDGETS)
print("Study:", STUDY_ID, f"({SCALE_PERCENT}% of the {BASELINE_ARCHIVES}-archive baseline)")


## Install dependencies and checkout immutable training code

In [ ]:
# Kaggle already supplies CUDA-enabled PyTorch.
%pip install -q torch-geometric "huggingface_hub[hf_xet]>=0.32" pyyaml pandas


In [ ]:
if (REPO_DIR / ".git").is_dir():
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "fetch", "--all", "--tags"],
        check=True,
    )
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REPO_REF], check=True)
commit = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
).strip()
assert commit == REPO_REF, (commit, REPO_REF)

sys.path.insert(0, str(REPO_DIR / "src"))
from ll_hls4ml.data.tensorize import EMBED_SIZE
from ll_hls4ml.io.schema import (
    BLOCK_FEATURE_SIZE,
    FUNCTION_FEATURE_SIZE,
    PRAGMA_FEATURE_SIZE,
)
from ll_hls4ml.data.vocab import load_vocab
from ll_hls4ml.models.registry import build, list_models

assert "hierarchical" in set(list_models()), (
    "REPO_REF does not contain the configured H0 model"
)
print("hls-surrogate-lab commit:", commit)
print("The split manifest will be generated from the current tensor index.")


## Build and validate the expanded official split metadata

In [ ]:
from threading import Event, Thread
import huggingface_hub
from huggingface_hub import HfApi, hf_hub_download, login, snapshot_download
from huggingface_hub.utils import enable_progress_bars

try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    hf_token = None
if hf_token:
    login(token=hf_token, add_to_git_credential=False)
enable_progress_bars()
print("huggingface_hub:", huggingface_hub.__version__)

HF_CACHE_DIR.mkdir(parents=True, exist_ok=True)
index_path = Path(hf_hub_download(
    repo_id=TENSOR_REPO_ID, filename="labels.json", repo_type="dataset",
    revision=TENSOR_REVISION, token=hf_token, cache_dir=str(HF_CACHE_DIR),
))
tensor_index = json.loads(index_path.read_text())
metadata = tensor_index.get("metadata", {})

def archive_number(path):
    return int(Path(path).parts[1].removeprefix("archive_"))

available_archives = sorted({
    archive_number(path) for path in tensor_index["labels"]
    if Path(path).parts[0] in KERNEL_TYPES
})
available_exemplar_archives = sorted({
    archive_number(path) for path in tensor_index["labels"]
    if Path(path).parts[0] == "exemplar"
})
assert available_archives, "Current tensor index has no main archives"
assert available_exemplar_archives, "Current tensor index has no exemplars"
family_archives = {
    family: sorted({
        archive_number(path) for path in tensor_index["labels"]
        if Path(path).parts[0] == family
    })
    for family in KERNEL_TYPES
}
expected_archives = list(range(1, EXPECTED_MAIN_ARCHIVES + 1))
assert available_archives == expected_archives, (
    f"Expected frozen archives {expected_archives}; found {available_archives}"
)
assert all(archives == expected_archives for archives in family_archives.values()), (
    f"Every kernel family must cover the frozen archive range: {family_archives}"
)
print("Current main archives:", available_archives)
print("Current exemplar archives:", available_exemplar_archives)

main_paths = sorted(
    path for path in tensor_index["labels"]
    if Path(path).parts[0] in KERNEL_TYPES
)
exemplar_paths = sorted(
    path for path in tensor_index["labels"]
    if Path(path).parts[0] == "exemplar"
)
all_paths = main_paths + exemplar_paths
assert all(len(Path(path).parts) == 3 and Path(path).suffix == ".pt" for path in all_paths)
graph_ids = [Path(path).stem for path in all_paths]
assert len(graph_ids) == len(set(graph_ids)), (
    "Duplicate graph/project IDs exist in the frozen tensor index"
)
assert set(all_paths) <= set(metadata), "Tensor index metadata is incomplete"
label_widths = {len(tensor_index["labels"][path]) for path in all_paths}
assert label_widths == {6}, f"Expected six targets per sample; got {label_widths}"
split_manifest = {name: [] for name in ("train", "validation", "test")}
for path in main_paths:
    split = str(metadata[path].get("dataset_split", "")).lower()
    split = "validation" if split in {"val", "validation"} else split
    assert split in split_manifest, f"Missing official split for {path}"
    split_manifest[split].append({
        "kernel_family": Path(path).parts[0], "tensor_path": path
    })
split_manifest["exemplar"] = [
    {"kernel_family": "exemplar", "tensor_path": path}
    for path in exemplar_paths
]
actual_sizes = {name: len(rows) for name, rows in split_manifest.items()}
assert all(actual_sizes[name] > 0 for name in split_manifest), actual_sizes
manifest_paths = [
    row["tensor_path"] for rows in split_manifest.values() for row in rows
]
assert len(manifest_paths) == len(set(manifest_paths)) == len(all_paths), (
    "Split manifest must contain every indexed sample exactly once"
)
CONFIG_DIR.mkdir(parents=True, exist_ok=True)
SPLIT_MANIFEST_PATH = CONFIG_DIR / "current_archives_manifest.json"
SPLIT_MANIFEST_PATH.write_text(json.dumps(split_manifest, indent=2))
tensor_paths = sorted(manifest_paths)
download_patterns = [
    f"{family}/archive_{archive}/*.pt"
    for family in (*KERNEL_TYPES, "exemplar")
    for archive in sorted({archive_number(path) for path in tensor_index["labels"] if Path(path).parts[0] in {*KERNEL_TYPES, "exemplar"}})
]
download_file_count = sum(
    1 for path in tensor_index["labels"]
    if Path(path).parts[0] in {*KERNEL_TYPES, "exemplar"}
)

def directory_size(path):
    total = 0
    if not path.exists():
        return total
    for item in path.rglob("*"):
        try:
            if item.is_file():
                total += item.stat().st_size
        except FileNotFoundError:
            pass  # A concurrent download may rename a temporary file.
    return total

def download_heartbeat(stop_event):
    previous = None
    last_change = time.monotonic()
    while not stop_event.wait(DOWNLOAD_HEARTBEAT_SECONDS):
        completed = len(list(HF_CACHE_DIR.rglob("*.pt")))
        visible_bytes = directory_size(HF_CACHE_DIR)
        state = (completed, visible_bytes)
        now = time.monotonic()
        if state != previous:
            last_change = now
            previous = state
        quiet_minutes = (now - last_change) / 60
        warning = (
            " WARNING: no visible disk progress"
            if quiet_minutes >= DOWNLOAD_STALL_WARNING_MINUTES else ""
        )
        print(
            f"[download heartbeat] {completed}/{download_file_count} tensor files; "
            f"{visible_bytes / 2**30:.2f} GiB visible; "
            f"unchanged {quiet_minutes:.1f} min.{warning}",
            flush=True,
        )

VOCAB_PATH = Path(hf_hub_download(
    repo_id=TENSOR_REPO_ID, filename="vocab.json", repo_type="dataset",
    revision=TENSOR_REVISION, token=hf_token, cache_dir=str(HF_CACHE_DIR),
))
TENSOR_DIR = index_path.parent
assert VOCAB_PATH.is_file()
print("Expanded split sizes:", actual_sizes)
print("Metadata snapshot:", TENSOR_DIR)
print("CDFG tensors are downloaded after preflight only if an active run needs them.")


## Validate the frozen H0 cohort and disclose official-split grouping conflicts

In [ ]:
def archive_set(split):
    return {Path(row["tensor_path"]).parts[1] for row in split_manifest[split]}

for split in ("train", "validation", "test"):
    assert archive_set(split) == {f"archive_{index}" for index in available_archives}
assert archive_set("exemplar") == {
    f"archive_{index}" for index in available_exemplar_archives
}
group_splits = {}
split_conflicts = []
for split in ("train", "validation", "test"):
    for row in split_manifest[split]:
        path = row["tensor_path"]
        group_id = str(metadata[path].get("group_id") or Path(path).stem)
        key = (row["kernel_family"], group_id)
        previous = group_splits.setdefault(key, split)
        if previous != split:
            split_conflicts.append({"key": key, "first_split": previous, "conflicting_split": split, "tensor_path": path})

cohort_audit = {
    "study_id": STUDY_ID,
    "protocol_id": "official_in_distribution",
    "archive_count": len(available_archives),
    "split_sizes": actual_sizes,
    "official_group_conflicts": split_conflicts,
    "grouping_warning": (
        "group_id is provenance only; this E1 result is not architecture-grouped"
    ),
}
COHORT_AUDIT_PATH = CONFIG_DIR / "cohort_audit.json"
COHORT_AUDIT_PATH.write_text(json.dumps(cohort_audit, indent=2))
print("Validated exact frozen cohort boundaries from metadata only.")
print("Official group conflicts retained and disclosed:", len(split_conflicts))
if split_conflicts:
    print(split_conflicts)


## Matched configurations and cross-session resume support

In [ ]:
common = {
    "tensor_dir": str(TENSOR_DIR),
    "tensor_source_revision": TENSOR_REVISION,
    "vocab_path": str(VOCAB_PATH),
    "split_manifest_path": str(SPLIT_MANIFEST_PATH),
    "require_complete_split_manifest": True,
    "results_dir": str(RESULTS_DIR),
    "kernel_types": KERNEL_TYPES,
    "study_id": STUDY_ID,
    "protocol_id": "official_in_distribution",
    "scale_percent": SCALE_PERCENT,
    "expected_main_archives": EXPECTED_MAIN_ARCHIVES,
    "distributed_world_size": GPU_COUNT if USE_DDP else 1,
    "seed": SEED,
    "family_balanced_sampling": False,
    "effective_global_batch": EFFECTIVE_GLOBAL_BATCH,
    "batch_size": EFFECTIVE_GLOBAL_BATCH // (GPU_COUNT if USE_DDP else 1),
    "num_workers": 2,
    "worker_tmpdir": "/tmp",
    "pin_memory": True,
    "prefetch_factor": 2,
    "thread_prefetch": False,
    "precision": PRECISION,
    "epochs": 400,
    "patience": 20,
    "learning_rate": 1e-3,
    "weight_decay": 1e-4,
    "hidden_dim": 64,
    "num_layers": 3,
    "dropout": 0.15,
    "use_global_features": True,
    "use_context": True,
    "context_mode": "core",
    "split_heads": True,
    "hurdle_heads": True,
    "hurdle_prediction_mode": "threshold",
    "loss": "log_huber_hurdle",
    "log_huber_delta": 0.35,
    "hurdle_classification_weight": 0.25,
    "early_stopping_metric": "smape",
    "gradient_clip_norm": 1.0,
    "lr_scheduler_patience": 8,
    "lr_scheduler_factor": 0.5,
    "min_learning_rate": 1e-6,
    "checkpoint_interval": 5,
    "verbose": 2,
}
experiments = {
    "hierarchical": f"h0_a{EXPECTED_MAIN_ARCHIVES}_seed{SEED}",
}
run_configs = {
    "hierarchical": {**common, "model": "hierarchical"},
}
for name, config in run_configs.items():
    experiment = experiments[name]
    config["experiment_name"] = experiment
    config["checkpoint_dir"] = str(RESULTS_DIR / experiment / "checkpoints")

CONFIG_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
_cached_previous_root = None

def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

SPLIT_MANIFEST_SHA256 = file_sha256(SPLIT_MANIFEST_PATH)

def previous_search_root():
    global _cached_previous_root
    if _cached_previous_root is not None:
        return _cached_previous_root
    root = (
        Path(PREVIOUS_RESULTS_ROOT)
        if PREVIOUS_RESULTS_ROOT is not None
        else Path("/kaggle/input")
    )
    archives = sorted(root.rglob(f"{ARCHIVE_NAME}.zip"))
    if not archives:
        assert PREVIOUS_RESULTS_ROOT is None, (
            f"No {ARCHIVE_NAME}.zip found under {root}"
        )
        return None
    if archives:
        assert len(archives) == 1, f"Multiple previous archives: {archives}"
        extracted = WORK / f"previous_{STUDY_ID}_results"
        if not extracted.is_dir():
            shutil.unpack_archive(archives[0], extracted)
        root = extracted
    _cached_previous_root = root
    return root

def resume_signature(name, config):
    keys = [
        "study_id", "protocol_id", "model", "scale_percent",
        "expected_main_archives", "seed", "effective_global_batch",
        "batch_size", "precision",
        "distributed_world_size",
        "epochs", "patience", "learning_rate", "weight_decay",
        "hidden_dim", "num_layers", "heads", "dropout",
        "high_level_encoder", "use_global_features", "use_context",
        "context_mode", "split_heads", "hurdle_heads", "loss",
        "gradient_clip_norm", "lr_scheduler_patience",
        "lr_scheduler_factor", "min_learning_rate",
        "checkpoint_interval",
    ]
    signature = {key: config.get(key) for key in keys}
    signature.update({
        "tensor_source_revision": TENSOR_REVISION,
        "split_manifest_sha256": SPLIT_MANIFEST_SHA256,
    })
    return signature

def find_previous_run(experiment):
    root = previous_search_root()
    if root is None:
        return None
    matches = sorted(
        path.parent for path in root.rglob("notebook_resume_signature.json")
        if path.parent.name == experiment
    )
    assert len(matches) <= 1, f"Multiple previous runs for {experiment}: {matches}"
    return matches[0] if matches else None

def prepare_run(name):
    config = run_configs[name]
    experiment = config["experiment_name"]
    run_dir = RESULTS_DIR / experiment
    expected = resume_signature(name, config)
    previous = find_previous_run(experiment)
    if not run_dir.exists() and previous is not None:
        prior = json.loads((previous / "notebook_resume_signature.json").read_text())
        prior.pop("repo_ref", None)
        assert prior == expected, f"Refusing incompatible resume for {experiment}"
        shutil.copytree(previous, run_dir)
        print("Imported previous run:", previous)
    run_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(COHORT_AUDIT_PATH, run_dir / "cohort_audit.json")
    signature_path = run_dir / "notebook_resume_signature.json"
    if signature_path.is_file():
        local_signature = json.loads(signature_path.read_text())
        local_signature.pop("repo_ref", None)
        assert local_signature == expected, (
            f"Local provenance changed for {experiment}"
        )
    signature_path.write_text(json.dumps(expected, indent=2))
    payload = dict(config)
    backup = Path(config["checkpoint_dir"]) / f"{experiment}_backup.pt"
    if backup.is_file():
        payload["resume_checkpoint_path"] = str(backup)
        print("Will resume", experiment, "from", backup)
    config_path = CONFIG_DIR / f"{experiment}.json"
    config_path.write_text(json.dumps(payload, indent=2))
    return config_path

for name in ACTIVE_RUNS:
    prepare_run(name)


## Download CDFG tensors only when needed, then train and package incrementally

In [ ]:
if "hierarchical" in ACTIVE_RUNS:
    remote_files = set(HfApi().list_repo_files(
        TENSOR_REPO_ID, repo_type="dataset", revision=TENSOR_REVISION,
        token=hf_token,
    ))
    missing_remote = sorted(set(tensor_paths) - remote_files)
    assert not missing_remote, (
        f"Frozen index references missing remote tensors; first: {missing_remote[:5]}"
    )
    representative_path = next(
        path for path in tensor_paths if Path(path).parts[0] != "exemplar"
    )
    representative_file = Path(hf_hub_download(
        repo_id=TENSOR_REPO_ID, filename=representative_path,
        repo_type="dataset", revision=TENSOR_REVISION, token=hf_token,
        cache_dir=str(HF_CACHE_DIR),
    ))
    sample = torch.load(representative_file, map_location="cpu", weights_only=False)
    assert "function" in sample.node_types and sample["function"].num_nodes > 0
    assert sample["instruction"].x.ndim == 2 and sample["instruction"].x.shape[1] == 1
    assert sample["function"].x.shape[1] == FUNCTION_FEATURE_SIZE
    assert sample["block"].x.shape[1] == BLOCK_FEATURE_SIZE
    assert sample["pragma"].x.shape[1] == PRAGMA_FEATURE_SIZE
    for node_type in ("variable", "constant"):
        assert sample[node_type].x.shape[1] == EMBED_SIZE
    assert getattr(sample, "hierarchy_schema_version", None) == 2
    for key in ("call_depth", "is_root", "is_entry", "is_reachable"):
        assert hasattr(sample["function"], key), f"Function hierarchy is missing {key}"
    for node_type in ("instruction", "block"):
        assert hasattr(sample[node_type], "call_depth")
    vocab, max_pos, _ = load_vocab(VOCAB_PATH)
    assert int(sample["instruction"].x.max()) < len(vocab)
    h0_config = run_configs["hierarchical"]
    smoke_model = build(
        "hierarchical", instruction_vocab_size=len(vocab),
        edge_pos_vocab_size=max_pos, y_means=torch.zeros(6),
        y_stds=torch.ones(6), hidden_dim=h0_config["hidden_dim"],
        num_layers=h0_config["num_layers"], dropout=h0_config["dropout"],
        use_global_features=h0_config["use_global_features"],
        use_context=h0_config["use_context"],
        context_mode=h0_config["context_mode"],
        split_heads=h0_config["split_heads"],
        hurdle_heads=h0_config["hurdle_heads"],
        hurdle_prediction_mode=h0_config["hurdle_prediction_mode"],
    ).eval()
    with torch.no_grad():
        smoke_output = smoke_model(sample)
    assert smoke_output.shape == (1, 8), smoke_output.shape
    del smoke_model, smoke_output, sample
    print("Preflight passed: remote membership, tensor schema, and H0 forward contract.")
    print(
        f"Downloading {download_file_count} tensors from "
        f"{TENSOR_REPO_ID} at {TENSOR_REVISION}.",
        flush=True,
    )
    print("A stdout heartbeat will appear once per minute.", flush=True)
    heartbeat_stop = Event()
    heartbeat_thread = Thread(
        target=download_heartbeat, args=(heartbeat_stop,), daemon=True
    )
    heartbeat_thread.start()
    try:
        TENSOR_DIR = Path(snapshot_download(
            repo_id=TENSOR_REPO_ID,
            repo_type="dataset",
            revision=TENSOR_REVISION,
            token=hf_token,
            cache_dir=str(HF_CACHE_DIR),
            allow_patterns=[*download_patterns, "labels.json", "vocab.json"],
        ))
    finally:
        heartbeat_stop.set()
        heartbeat_thread.join()
    print("Download call completed.", flush=True)
    missing_tensors = [path for path in tensor_paths if not (TENSOR_DIR / path).is_file()]
    assert not missing_tensors, f"Missing tensors; first: {missing_tensors[:5]}"
    print("Validated complete local tensor membership.")
else:
    print("H0 is inactive: skipping the CDFG tensor download.")

import shlex

TRAIN_SCRIPT = REPO_DIR / "scripts/train.py"
run_environment = os.environ.copy()
run_environment["PYTHONPATH"] = str(REPO_DIR / "src")
run_environment["LL_HLS4ML_TQDM"] = "0"
run_environment["MPLCONFIGDIR"] = "/kaggle/working/matplotlib"

def base_training_command(config_path):
    if USE_DDP and GPU_COUNT > 1:
        return [
            sys.executable, "-m", "torch.distributed.run", "--standalone",
            f"--nproc_per_node={GPU_COUNT}", str(TRAIN_SCRIPT),
            "--config", str(config_path),
        ]
    return [sys.executable, str(TRAIN_SCRIPT), "--config", str(config_path)]

def run_and_stream(command, log_path):
    print("$", shlex.join(command), flush=True)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open("a", buffering=1) as log:
        process = subprocess.Popen(
            command, cwd=REPO_DIR, env=run_environment,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1,
        )
        for line in process.stdout:
            print(line, end="")
            log.write(line)
        return process.wait()

def package_results():
    archive = Path(shutil.make_archive(
        str(WORK / ARCHIVE_NAME), "zip", root_dir=RESULTS_DIR
    ))
    print("Updated result archive:", archive)
    return archive

def run_experiment(name):
    if name not in ACTIVE_RUNS:
        print("Skipping", name, "by ACTIVE_RUNS configuration.")
        return
    config = run_configs[name]
    config_path = prepare_run(name)
    experiment = config["experiment_name"]
    run_dir = RESULTS_DIR / experiment
    summary_path = run_dir / "summary.json"
    if summary_path.is_file():
        summary = json.loads(summary_path.read_text())
        timeout_evaluation = summary.get("resolved_config", {}).get(
            "evaluation_checkpoint_path"
        )
        if timeout_evaluation is None:
            print(experiment, "already completed normally; skipping.")
            package_results()
            return
        print(experiment, "has a timeout evaluation; resuming training.")
    command = [
        "timeout", "--signal=INT", "--kill-after=5m", TRAIN_BUDGETS[name],
        *base_training_command(config_path),
    ]
    try:
        started = time.time()
        return_code = run_and_stream(command, run_dir / "training.log")
        print(experiment, "return code:", return_code)
        print(experiment, "wall seconds:", round(time.time() - started, 1))
        if return_code == 0:
            assert summary_path.is_file(), "Training exited successfully without a summary"
            print("Training and evaluation completed normally.")
            return
        assert return_code in {124, 130, 137}, (
            f"Training failed with unexpected return code {return_code}; "
            "refusing to hide the failure behind checkpoint evaluation"
        )
        best_checkpoint = (
            Path(config["checkpoint_dir"]) / f"{experiment}_checkpoint.pt"
        )
        if not best_checkpoint.is_file():
            best_checkpoint = (
                Path(config["checkpoint_dir"]) / f"{experiment}_backup.pt"
            )
        assert best_checkpoint.is_file(), (
            "The time limit expired before a checkpoint existed. "
            "Use a budget long enough to reach the checkpoint cadence."
        )
        evaluation_command = [
            sys.executable, str(TRAIN_SCRIPT), "--config", str(config_path),
            "--evaluate-checkpoint", str(best_checkpoint),
        ]
        evaluation_code = run_and_stream(
            evaluation_command, run_dir / "evaluation.log"
        )
        assert evaluation_code == 0, f"Evaluation failed: {evaluation_code}"
    finally:
        package_results()


## Run the final expanded-data H0 reference

Run this cell in every session. It skips a normally completed run and resumes
a time-capped run when its optimizer backup is present.

In [ ]:
run_experiment("hierarchical")


## Compare available results and download the resume archive

In [ ]:
import numpy as np
import pandas as pd

TARGET_NAMES = ("lut", "ff", "dsp", "bram", "cycles_max", "interval_max")
current_paths = {
    split: {row["tensor_path"] for row in split_manifest[split]}
    for split in ("test", "exemplar")
}

def prediction_metrics(frame):
    smapes = []
    r2s = []
    for target in TARGET_NAMES:
        truth = frame[f"target_{target}"].to_numpy(dtype=float)
        prediction = frame[f"prediction_{target}"].to_numpy(dtype=float)
        denominator = np.abs(truth) + np.abs(prediction) + 1.0
        smape = 200 * np.abs(prediction - truth) / denominator
        residual = np.square(truth - prediction).sum()
        total = np.square(truth - truth.mean()).sum()
        smapes.append(smape.mean())
        r2s.append(1 - residual / total if total > 0 else float("nan"))
    return float(np.mean(smapes)), float(np.nanmean(r2s))

rows = []
for name, experiment in experiments.items():
    run_dir = RESULTS_DIR / experiment
    metrics_path = run_dir / "metrics.csv"
    summary_path = run_dir / "summary.json"
    if not (metrics_path.is_file() and summary_path.is_file()):
        continue
    metrics = pd.read_csv(metrics_path)
    metrics = metrics[metrics["kernel_family"] == "all"]
    summary = json.loads(summary_path.read_text())
    for split in ("test", "exemplar"):
        selected = metrics[metrics["split"] == split]
        rows.append({
            "model": name,
            "split": split,
            "macro_smape": selected["smape"].mean(),
            "macro_r2": selected["r2"].mean(),
            "best_epoch": summary.get("best_epoch"),
            "best_validation_smape": summary.get("best_metric"),
            "training_samples": summary["sizes"]["train"],
            "time_capped": bool(
                summary.get("resolved_config", {}).get("evaluation_checkpoint_path")
            ),
        })
    predictions = pd.read_csv(run_dir / "predictions.csv")
    for split in ("test", "exemplar"):
        selected = predictions[
            (predictions["split"] == split)
            & predictions["tensor_path"].isin(current_paths[split])
        ]
        assert len(selected) == len(current_paths[split]), (
            name, split, len(selected), len(current_paths[split])
        )
        macro_smape, macro_r2 = prediction_metrics(selected)
        rows.append({
            "model": name,
            "split": f"{split}_current_archives",
            "macro_smape": macro_smape,
            "macro_r2": macro_r2,
            "best_epoch": summary.get("best_epoch"),
            "best_validation_smape": summary.get("best_metric"),
            "training_samples": summary["sizes"]["train"],
            "time_capped": bool(
                summary.get("resolved_config", {}).get("evaluation_checkpoint_path")
            ),
        })
comparison = pd.DataFrame(rows)
display(comparison)
if not comparison.empty:
    comparison.to_csv(RESULTS_DIR / f"{STUDY_ID}_summary.csv", index=False)
archive = package_results()
from IPython.display import FileLink
display(FileLink(str(archive)))


## Interpretation guardrails

- A time-capped result is provisional even though the best checkpoint is evaluated.
- Treat this as the E1 in-distribution reference, not evidence of architecture induction.
- Compare runs only when their split hash, tensor revision, and exact prediction membership match.
- `group_id` is not a validated architecture signature; disclosed official-split conflicts are not repaired post hoc.
- This seed-42 run is the preregistered H0 reference. Additional seeds belong to the later grouped structural contrast.
- Keep the produced zip: it contains all periodic optimizer backups needed for the next Kaggle session.
